# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library, referencing all entities by their `@id` fields as per the Croissant specification.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.
We enumerate the record sets and for each, list available fields (columns), always referencing by `@id`.

In [ ]:
# List all record sets and their fields by @id
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets declared in the top-level metadata. Attempting to infer from distribution data...")
    # Some datasets list record sets in metadata.distribution, we'll list distributions
    for dist in getattr(dataset.metadata, 'distributions', []) + getattr(dataset.metadata, 'distribution', []):
        print(f"  Distribution: @id={getattr(dist, '@id', None)}, name={getattr(dist, 'name', None)}")
else:
    for rs in record_sets:
        print(f"RecordSet: @id={rs['@id']}, name={rs.get('name', 'Unknown')}")
        fields = rs.get('field', []) or []
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            print(f"  Field: @id={field['@id']}, name={field.get('name', 'Unknown')}")

# Alternatively, show available record set IDs using mlcroissant API where possible:
rs_ids = []
for rs in dataset.record_sets:
    print(f"Record Set: @id={rs['@id']}, name={rs.get('name', 'Unknown')}")
    rs_ids.append(rs['@id'])
    for field in rs.get('field', []):
        if isinstance(field, dict) and '@id' in field:
            print(f"  Field: @id={field['@id']}, name={field.get('name', 'Unknown')}")
        elif isinstance(field, list):
            for f in field:
                print(f"  Field: @id={f['@id']}, name={f.get('name', 'Unknown')}")
if not rs_ids:
    print("No explicit record set found in metadata; you may need to specify the record set id manually based on fileObject/distribution.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 

**Note**: All field and record set references use their `@id` as per the Croissant metadata. Fill these in based on output above if necessary.

Below we attempt to enumerate and load all record sets. You may need to adjust the `record_sets_ids` variable depending on the record set structure.

In [ ]:
# ---
# Attempt to enumerate all record sets by @id
record_sets = getattr(dataset.metadata, 'record_sets', [])
if not record_sets:  # fallback
    # Try to infer file objects from distribution
    record_sets = [r for r in getattr(dataset.metadata, 'distributions', []) + getattr(dataset.metadata, 'distribution', [])]

record_set_ids = []
for rs in getattr(dataset, 'record_sets', []):
    record_set_ids.append(rs['@id'])

# If none are found, attempt to use a default ID (replace as needed)
if not record_set_ids:
    # Replace this with the actual record set @id from the dataset overview
    record_set_ids = [
        # Example: 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034',
    ]
    print("Please update 'record_set_ids' above with the actual '@id' for the record set you want to load.")

dataframes = {}
for rs_id in record_set_ids:
    try:
        df = pd.DataFrame(list(dataset.records(record_set=rs_id)))
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for record_set @id={rs_id}, shape={df.shape}")
        print(df.head(2))
    except Exception as e:
        print(f"Could not load records for record_set @id={rs_id}: {e}")

# For demonstration purposes, select the first record set for analysis
if record_set_ids:
    example_record_set_id = record_set_ids[0]
else:
    example_record_set_id = None

if example_record_set_id:
    print(f"\nAvailable fields in record set @id={example_record_set_id}:\n{dataframes[example_record_set_id].columns.tolist()}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

In this example, we select a numeric field (using its `@id` as column name), filter the DataFrame, normalize it, and group by another field (also by `@id`).

**Remember:** Replace `<numeric_field_id>` and `<group_field_id>` below with actual `@id` values of relevant fields/columns from your dataset. Use the output of section 3 to guide your choice.

In [ ]:
from pandas.api.types import is_numeric_dtype

df = dataframes.get(example_record_set_id)
if df is not None and not df.empty:
    # Try to find a numeric column by @id (just picking one as an example)
    numeric_field_id = None
    for col in df.columns:
        if is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        print("No numeric field found for analysis. Please update 'numeric_field_id' below with an actual numeric @id from this record set.")
        numeric_field_id = df.columns[0]  # fallback for demonstration
    print(f"Using numeric field @id: {numeric_field_id}")

    # Filter records: in mass-spectrometry, could filter by a minimum abundance threshold, etc.
    threshold = df[numeric_field_id].mean() if is_numeric_dtype(df[numeric_field_id]) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head(3))

    # Normalization (z-score)
    colname_norm = f"{numeric_field_id}_normalized"
    filtered_df[colname_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, colname_norm]].head(3))

    # Grouping by another field
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and df[col].nunique() < 10:
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field_id} (using @id):")
        print(grouped_df.head())
    else:
        print("No suitable group field found. Update 'group_field_id' for your dataset.")
else:
    print("No data loaded for EDA. Confirm record set extraction above and try again.")

## 5. Visualization
Visualize distributions or relationships between fields in the selected record set. You may update the field `@id`s below based on available columns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for the chosen numeric field
if df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# If a group field is available, plot boxplot
if df is not None and group_field_id in df.columns and numeric_field_id in df.columns:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()
else:
    print("No suitable group field for boxplot. Assign 'group_field_id' to an appropriate @id.")

## 6. Conclusion
This notebook demonstrated how to load and explore a dataset described by the Croissant schema using the `mlcroissant` library.

Key steps included:
- Loading the metadata and reviewing dataset information.
- Enumerating available record sets and fields using `@id`.
- Extracting records into Pandas DataFrames, referencing record sets and fields by their `@id`.
- Performing exploratory analysis and basic visualizations on numeric fields.

**Remember:** Always reference Croissant entities via their unique `@id`. For custom analysis, review available fields using the overview steps and substitute the `@id`s as necessary for deeper exploration.

For additional information on the dataset or schema, see the [FAIR^2 Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).
